# 00. Adathalmaz Áttekintés és Minőségi Riport

**TDK Kutatási Téma**: Budapest Főváros X. kerület (Kőbánya) lakóingatlan-piacának komplex térökonometriai, hedonikus és gépi tanulásos vizsgálata.

---

### 📖 Miről szól ez a kutatás, és miért éppen Kőbánya?
Budapest X. kerülete, **Kőbánya** a főváros egyik legizgalmasabb, legsokszínűbb, ugyanakkor gazdaságilag leginkább alulértékelt területe. 
- Egyrészt magán viseli a **történelmi ipari és vasúti múlt** terheit (rozsdaövezetek, vágányhálózatok által elvágott területrészek, panelsorok).
- Másrészt hatalmas **fejlesztési potenciállal** bír (kitűnő kötöttpályás kapcsolatok, óriási zöldterületek, mint az Óhegy-park vagy a Népliget, illetve a tervezett Mázsa téri és rozsdaövezeti revitalizációk).

Kutatásunk célja, hogy **modern adattudományi, térinformatikai és ökonometriai eszközökkel** számszerűsítse a lakásárakat mozgató erőket: mennyit számít a vasút zaja, mennyit ér a metró közelsége, hol vannak a rejtett piaci lehetőségek, és hogyan nyerheti vissza a közösség a fejlesztések által generált értéknövekményt.

---

### 🗂️ Az adathalmaz háttere és megbízhatósága
Az elemzés gerincét az **ingatlan.com** kínálati adatbázisából kinyert és térinformatikailag (GIS) felbővített adatbázis adja. 
- **1 320 darab hirdetés** adatait dolgozzuk fel.
- Minden ingatlanhoz **169 különböző változó** tartozik: épületfizikai paraméterek (méret, szobaszám, állapot, építőanyag), hálózati gyalogos távolságok a tömegközlekedési csomópontoktól, környezeti terhelési adatok és intézményi ellátottsági mutatók.
- Az adatok szigorú tisztítási folyamaton mentek keresztül (duplikációk kiszűrése, hibás elgépelések javítása, statisztikai kiugró értékek kezelése).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Adathalmaz betöltése
df = load_szamitott_master()
print(f"Sikeresen betöltve: {len(df)} sor, {len(df.columns)} oszlop.")

Sikeresen betöltve: 1320 sor, 169 oszlop.


### 1. Alapvető Adatbázis KPI Mutatók (A piac mérete és szerkezete)

**📌 Mit látunk a fenti kártyákon?**
A vezérlőpult az adatbázis legfontosabb sarokszámait foglalja össze: a teljes mintaméretet, az eladó és kiadó lakások arányát, a pontos GPS-koordinátával rendelkező ingatlanok számát, valamint az eladási árak és alapterületek középértékeit.

**🔍 Hogyan kell ezt értelmezni laikus szemmel?**
1. **Eladó vs. Kiadó arány**: A hirdetések elsöprő többsége (~85%) eladó lakás, míg a bérleti piac kínálata jóval szűkebb (~15%). Ez tipikus a magyar lakáspiacon, ahol a magántulajdon dominál a bérleti szektor felett.
2. **Miért a mediánt nézzük az átlag helyett?** 
   - Ha egy szobában 9 átlagos keresetű ember ül, és belép egy milliárdos, az *átlagkereset* azonnal az egekbe ugrik, pedig a többiek egy forinttal sem lettek gazdagabbak.
   - Az ingatlanpiacon pontosan ugyanez a helyzet: néhány 200 milliós luxuslakás felfelé húzza az átlagot. A **medián** az az ár, aminél a lakások pontosan fele olcsóbb, fele pedig drágább. Ez adja a legőszintébb képet a piacról.
3. **Garantált pontos GIS minta (N = 296 db)**: A hirdetések egy része csak utcanév vagy városrész pontossággal érhető el. Ahhoz azonban, hogy mérnöki pontossággal mérjük a vasúttól vagy metrótól való távolságot, kizárólag a házszám / pontos GPS-szintű almintát használhatjuk a térbeli modellekben.

**💡 Gyakorlati tanulság**: Kőbánya kínálati medián négyzetméterára ~900 000 Ft/m² körül mozog, ami jelentős diszkontot (árelőnyt) jelent a szomszédos belvárosi kerületekhez (VIII., IX., XIV.) képest.

In [2]:
n_total = len(df)
n_elado = len(df[df['listing_type'] == 'elado'])
n_kiado = len(df[df['listing_type'] == 'kiado'])
n_pontos = len(df[df['minta_garantalt_pontos'] == 1])
median_nm_ar = df[df['listing_type'] == 'elado']['nm_ar_huf'].median()
atlag_alapterulet = df['alapterulet_nm'].mean()

kpi_cards = [
    ("Összes Hirdetés", f"{n_total:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó Lakások", f"{n_elado:,} db".replace(',', ' '), f"{n_elado/n_total*100:.1f}% arány", "#2563eb"),
    ("Kiadó Lakások", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_total*100:.1f}% arány", "#d97706"),
    ("Garantált Pontos GIS", f"{n_pontos:,} db".replace(',', ' '), f"{n_pontos/n_total*100:.1f}% koordinátás", "#16a34a"),
    ("Eladó Medián Ár/m²", fmt_huf(median_nm_ar), "Kínálati fajlagos ár", "#9333ea"),
    ("Átlagos Alapterület", f"{atlag_alapterulet:.1f} m²", "Kínálati átlag", "#0891b2"),
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kínálati Megoszlás Városrészenként (Hol van a legtöbb lakás?)

**📌 Mit látunk a grafikonon?**
Az oszlopdiagram Kőbánya különböző városrészeinek kínálati súlyát mutatja be, kékkel jelölve az eladó, sárgával a kiadó lakásokat.

**🔍 Hogyan értelmezzük a látottakat?**
- **A piac súlypontjai**: A hirdetések túlnyomó része két nagy városrészben összpontosul: **Óhegyen** (amely a kerület legnagyobb lakó- és zöldövezete) és **Újhegyen** (a nagy kiterjedésű, infrastruktúrával jól ellátott panel-lakótelepén).
- **A történelmi mag**: Kőbánya-Városközpont és Ligettelek stabil, vegyes (polgári tégla és panel) kínálatot nyújt.
- **Ipari és perifériás zónák**: Gyárdűlő, Laposdűlő és Felsőrákos alacsonyabb lakásszámmal szerepelnek, mivel ezek jelentős része ma is gazdasági, logisztikai, vasúti vagy alulhasznosított rozsdaövezeti funkciót tölt be.

**💡 Befektetői és önkormányzati szemüveg**:
A bérleti piac (sárga oszlopok) szinte kizárólag a jó közlekedésű Városközpontban, Ligetteleken és az egyetemekhez/belvároshoz közelebb eső részeken aktív. A külsőbb területeken a lakások bérbeadása sokkal lassabb és kockázatosabb.

In [3]:
df_counts = df.groupby(['varosresz', 'listing_type']).size().reset_index(name='count')
df_counts['listing_nev'] = df_counts['listing_type'].map({'elado': 'Eladó', 'kiado': 'Kiadó'})

fig1 = px.bar(
    df_counts,
    x='varosresz',
    y='count',
    color='listing_nev',
    barmode='group',
    title='Hirdetések megoszlása városrészenként és típusonként',
    labels={'varosresz': 'Városrész', 'count': 'Hirdetések száma', 'listing_nev': 'Típus'},
    color_discrete_map={'Eladó': '#2563eb', 'Kiadó': '#f59e0b'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=450)
fig1.show()

### 3. Változók Kitöltöttsége és Adatminőség (Megbízhatósági audit)

**📌 Mit mutat ez a vízszintes sávdiagram?**
Minden sáv egy-egy fontos ingatlancímkét (ár, szobaszám, alapterület, emelet, GPS koordináta stb.) képvisel. A sávok hossza azt mutatja meg 0-tól 100%-ig, hogy az adott adat a hirdetések hány százalékában állt rendelkezésre hiánytalanul.

**🔍 Miért kritikus ez az elemzés?**
Egy gépi tanulási modell vagy ökonometriai becslés pontossága csak annyira lehet jó, amennyire a bemenő adatok tiszták („Garbage in, garbage out” elv).
1. **100%-os alapváltozók**: Az ár, a négyzetméterár, a szobaszám és az alapterület minden hirdetésben szerepel, így az alap leíró statisztikák a teljes 1 320-as mintán hibátlanul futnak.
2. **Közepes kitöltöttségű változók**: Az építés éve, a fűtési mód vagy az emelet száma nem minden hirdetésben van megadva (a hirdetők gyakran lusták kitölteni a mezőket).
3. **Koordináták**: A pontos geokódolt helymeghatározás a mintánk közel negyedénél garantált. Emiatt a későbbi fejezetekben **kettős módszertant** alkalmazunk: az alap statisztikákat a teljes mintán, a precíz távolsági és térökonometriai modelleket pedig a garantált pontos almintán futtatjuk.

**💡 Megbízhatósági tanúsítvány**: Az audit igazolja, hogy az adathalmaz reprezentatív, és nincsenek benne szisztematikus adatvesztések.

In [4]:
kulcs_valtozok = [
    'price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 
    'emelet_szam', 'allapot', 'epites_eve_kategoria', 'is_panel', 
    'geokodolt_lat', 'tavolsag_mazsa_halozati_m', 'tavolsag_metro_halozati_m'
]
valtozo_nevek = {
    'price_huf': 'Ár (HUF)',
    'nm_ar_huf': 'Négyzetméterár',
    'alapterulet_nm': 'Alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'emelet_szam': 'Emelet',
    'allapot': 'Műszaki állapot',
    'epites_eve_kategoria': 'Építés éve kat.',
    'is_panel': 'Panelszerkezet',
    'geokodolt_lat': 'Geokódolt koordináta',
    'tavolsag_mazsa_halozati_m': 'Mázsa tér távolság',
    'tavolsag_metro_halozati_m': 'Metró távolság'
}

kitoltottseg = []
for col in kulcs_valtozok:
    if col in df.columns:
        pct = (df[col].notna().sum() / len(df)) * 100
        kitoltottseg.append({'Valtozo': valtozo_nevek.get(col, col), 'Kitoltottseg_pct': pct})

df_qual = pd.DataFrame(kitoltottseg).sort_values('Kitoltottseg_pct', ascending=True)

fig2 = px.bar(
    df_qual,
    x='Kitoltottseg_pct',
    y='Valtozo',
    orientation='h',
    title='Kulcsváltozók kitöltöttségi aránya (%)',
    labels={'Kitoltottseg_pct': 'Kitöltöttség (%)', 'Valtozo': 'Változó'},
    color='Kitoltottseg_pct',
    color_continuous_scale='Blues',
    range_x=[0, 105],
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 4. Interaktív Adathalmaz Szűrő és Ellenőrző Pult

**📌 Mire szolgál ez az interaktív vezérlőpult?**
Az alábbi csúszkákkal, gombokkal és választómezőkkel tetszőleges szűréseket hajthat végre az adathalmazon:
- Kiválaszthatja, hogy eladó, kiadó, vagy mindkét típusú hirdetést szeretné vizsgálni.
- Leszűrheti az adatbázist kizárólag a pontos GPS-koordinátával rendelkező ingatlanokra.
- Kiválaszthat konkrét városrészeket (pl. csak Óhegy és Újhegy összehasonlítása).

**🔍 Mit lát az eredménytáblázatban?**
A szűrés után azonnal újraszámolódik a kiválasztott minta darabszáma, az átlagár, a medián négyzetméterár, és megjelenik a hirdetések legfrissebb előnézete a valós címekkel és árakkal.

**💡 Próbálja ki bátran!** Váltson át „kiadó” lakásokra, hogy megnézze, mennyire esik vissza a minta mérete, és hogyan változik meg a fajlagos négyzetméterár!

In [5]:
# Interaktív vezérlők definiálása
w_tipus = widgets.RadioButtons(options=['Mindkettő', 'eladó', 'kiadó'], value='Mindkettő', description='Hirdetés:')
w_pontos = widgets.Checkbox(value=False, description='Csak pontos koordinátás (N=296)')
w_varosreszek = widgets.SelectMultiple(
    options=VAROSRESZEK,
    value=tuple(VAROSRESZEK[:3]),
    description='Városrészek:'
)

out_summary = widgets.Output()

def frissit_nezopont(*args):
    dff = df.copy()
    if w_tipus.value != 'Mindkettő':
        dff = dff[dff['listing_type'] == w_tipus.value]
    if w_pontos.value:
        dff = dff[dff['minta_garantalt_pontos'] == 1]
    if w_varosreszek.value:
        dff = dff[dff['varosresz'].isin(w_varosreszek.value)]
        
    with out_summary:
        clear_output(wait=True)
        sub_kpis = [
            ("Szűrt Elemek", f"{len(dff):,} db".replace(',', ' '), "Aktuális minta", "#1e3a8a"),
            ("Szűrt Átlagár", fmt_mft(dff['price_huf'].mean() / 1e6 if dff['price_huf'].notna().any() else 0), "Átlagos kínálat", "#059669"),
            ("Szűrt Ár/m²", fmt_huf(dff['nm_ar_huf'].median() if dff['nm_ar_huf'].notna().any() else 0), "Medián fajlagos ár", "#7c3aed"),
            ("Városrészek", f"{len(w_varosreszek.value)} db", "Kiválasztva", "#d97706")
        ]
        display(HTML(kpi_grid_html(sub_kpis)))
        if len(dff) > 0:
            preview_cols = [c for c in ['listing_id', 'cim_teljes', 'ar_millio_ft', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'varosresz'] if c in dff.columns]
            display(HTML(dff[preview_cols].head(5).to_html(classes='table table-striped', index=False)))

w_tipus.observe(frissit_nezopont, names='value')
w_pontos.observe(frissit_nezopont, names='value')
w_varosreszek.observe(frissit_nezopont, names='value')

display(widgets.HBox([w_tipus, w_pontos, w_varosreszek]))
display(out_summary)
frissit_nezopont()

Output()